In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/csiro-biomass/sample_submission.csv
/kaggle/input/csiro-biomass/train.csv
/kaggle/input/csiro-biomass/test.csv
/kaggle/input/csiro-biomass/test/ID1001187975.jpg
/kaggle/input/csiro-biomass/train/ID2099464826.jpg
/kaggle/input/csiro-biomass/train/ID2037861084.jpg
/kaggle/input/csiro-biomass/train/ID1211362607.jpg
/kaggle/input/csiro-biomass/train/ID1853508321.jpg
/kaggle/input/csiro-biomass/train/ID193102215.jpg
/kaggle/input/csiro-biomass/train/ID698608346.jpg
/kaggle/input/csiro-biomass/train/ID1859251563.jpg
/kaggle/input/csiro-biomass/train/ID1880764911.jpg
/kaggle/input/csiro-biomass/train/ID853954911.jpg
/kaggle/input/csiro-biomass/train/ID1403107574.jpg
/kaggle/input/csiro-biomass/train/ID1781353117.jpg
/kaggle/input/csiro-biomass/train/ID384648061.jpg
/kaggle/input/csiro-biomass/train/ID1563418511.jpg
/kaggle/input/csiro-biomass/train/ID2125100696.jpg
/kaggle/input/csiro-biomass/train/ID482555369.jpg
/kaggle/input/csiro-biomass/train/ID638711343.jpg
/kaggle/input/c

In [2]:
# ============================================================================
# CELL 1: Setup and Data Loading
# ============================================================================
import numpy as np
import pandas as pd
import os
import torch
import torchvision
import torch.nn as nn
import pytorch_lightning as pl
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision.transforms import v2
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from pytorch_lightning import Trainer
from tqdm import tqdm
import cv2
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint

# Load data
train_file_path = "/kaggle/input/csiro-biomass/train.csv"
test_file_path = "/kaggle/input/csiro-biomass/test.csv"

train_pd = pd.read_csv(train_file_path)
test_pd_original = pd.read_csv(test_file_path)  # Keep original for submission
test_pd = test_pd_original.copy()  # Working copy for predictions

print(f"Train shape: {train_pd.shape}")
print(f"Test shape: {test_pd.shape}")

Train shape: (1785, 9)
Test shape: (5, 3)


In [3]:
# ============================================================================
# CELL 2: HEIGHT PREDICTION
# ============================================================================
print("\n=== STEP 1: Height Prediction ===")

height_model = torchvision.models.detection.maskrcnn_resnet50_fpn(weights=None, weights_backbone=None)
path = "/kaggle/input/csiro-biomass"
local_weights = "/kaggle/input/mask-rcnn-models/pytorch/default/10/Maskrcnn_best.pt"

try:
    state_dict = torch.load(local_weights, map_location="cpu", weights_only=False)
    height_model = state_dict
    height_model.eval()
    
    for index, image_path in test_pd["image_path"].items():
        image_path = os.path.join(path, image_path)
        image = Image.open(image_path).convert("RGB")
        
        image_transform = v2.Compose([
            v2.Resize((224, 224)),
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])
        
        image_tensor = image_transform(image)
        
        with torch.no_grad():
            outputs = height_model([image_tensor])
        
        output_pixels = outputs[0]["boxes"].cpu().numpy()
        x1, y1, x2, y2 = output_pixels[0]
        image_in_cm = y2 / 2.54
        test_pd.loc[index, "Height_Ave_cm"] = image_in_cm
    
    print("✅ Height predictions completed using Mask R-CNN")
except:
    # Fallback to mean height
    mean_height = train_pd["Height_Ave_cm"].mean()
    test_pd["Height_Ave_cm"] = mean_height
    print(f"✅ Height predictions using mean: {mean_height:.2f}")


=== STEP 1: Height Prediction ===
✅ Height predictions completed using Mask R-CNN


species model

In [4]:
# ============================================================================
# CELL 3: SPECIES PREDICTION
# ============================================================================
print("\n=== STEP 2: Species Prediction ===")

# Global Encoders
SPECIES_LE = LabelEncoder()
TARGET_LE = LabelEncoder()

SPECIES_LE.fit(train_pd["Species"].astype(str).unique())
TARGET_LE.fit(train_pd["target_name"].astype(str).unique())

def safe_encode(le, val):
    """Encodes labels; returns 0 if label is unseen."""
    val_str = str(val)
    if val_str in le.classes_:
        return le.transform([val_str])[0]
    return 0

class SpeciesDataset(Dataset):
    def __init__(self, df, root_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = os.path.join(self.root_dir, row["image_path"])
        
        try:
            image = Image.open(image_path).convert("RGB")
        except:
            image = Image.new('RGB', (224, 224), (0, 0, 0))
            
        tabular = torch.tensor([
            float(row["target_name"]), 
            float(row["Height_Ave_cm"])
        ], dtype=torch.float32)
        
        y = torch.tensor(int(row["Species"]), dtype=torch.long)
        
        if self.transform:
            image = self.transform(image)
        return image, tabular, y

class SpeciesDataModule(pl.LightningDataModule):
    def __init__(self, train_df, valid_df, root_dir, batch_size=16, num_workers=2):
        super().__init__()
        self.train_df, self.valid_df = train_df.copy(), valid_df.copy()
        self.root_dir, self.batch_size, self.num_workers = root_dir, batch_size, num_workers

    def setup(self, stage=None):
        for df in [self.train_df, self.valid_df]:
            if not np.issubdtype(df["Species"].dtype, np.number):
                df["Species"] = df["Species"].apply(lambda x: safe_encode(SPECIES_LE, x))
            if not np.issubdtype(df["target_name"].dtype, np.number):
                df["target_name"] = df["target_name"].apply(lambda x: safe_encode(TARGET_LE, x))

        self.train_tf = v2.Compose([
            v2.RandomResizedCrop(224), v2.RandomHorizontalFlip(), v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])
        self.valid_tf = v2.Compose([
            v2.Resize(256), v2.CenterCrop(224), v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])
        
        self.train_ds = SpeciesDataset(self.train_df, self.root_dir, self.train_tf)
        self.valid_ds = SpeciesDataset(self.valid_df, self.root_dir, self.valid_tf)

    def train_dataloader(self):
        return DataLoader(self.train_ds, batch_size=self.batch_size, shuffle=True, num_workers=self.num_workers)
    
    def val_dataloader(self):
        return DataLoader(self.valid_ds, batch_size=self.batch_size, shuffle=False, num_workers=self.num_workers)

class EfficientNetSpeciesClassifier(pl.LightningModule):
    def __init__(self, num_classes, target_dim, lr=1e-4):
        super().__init__()
        self.save_hyperparameters()
        
        self.base_model = models.efficientnet_b0(weights=None)
        
        local_weights = "/kaggle/input/tf-efficientnet/pytorch/tf-efficientnet-b0/1/tf_efficientnet_b0_aa-827b6e33.pth"
        if os.path.exists(local_weights):
            state_dict = torch.load(local_weights, map_location="cpu")
            new_state_dict = {}
            # Mapping TF-style keys to PyTorch-style keys
            for k, v in state_dict.items():
                n = k.replace("conv_stem", "features.0.0").replace("bn1", "features.0.1")
                if "blocks" in n:
                    p = n.split(".")
                    b_idx = int(p[1]) + 1
                    sub = ".".join(p[2:])
                    if b_idx == 1:
                        sub = sub.replace("conv_dw", "block.0.0").replace("bn1", "block.0.1")
                        sub = sub.replace("se.conv_reduce", "block.1.fc1").replace("se.conv_expand", "block.1.fc2")
                        sub = sub.replace("conv_pw", "block.2.0").replace("bn2", "block.2.1")
                    else:
                        sub = sub.replace("conv_pw", "block.0.0").replace("bn1", "block.0.1")
                        sub = sub.replace("conv_dw", "block.1.0").replace("bn2", "block.1.1")
                        sub = sub.replace("se.conv_reduce", "block.2.fc1").replace("se.conv_expand", "block.2.fc2")
                        sub = sub.replace("conv_pwl", "block.3.0").replace("bn3", "block.3.1")
                    n = f"features.{b_idx}.{sub}"
                n = n.replace("conv_head", "features.8.0").replace("bn2", "features.8.1")
                new_state_dict[n] = v
            self.base_model.load_state_dict(new_state_dict, strict=False)
            print("✅ Loaded EfficientNet-B0 weights for Species Prediction")

        self.img_dim = self.base_model.classifier[1].in_features
        self.base_model.classifier = nn.Identity()

        self.target_emb = nn.Embedding(target_dim + 1, 8)
        self.tabular_net = nn.Sequential(
            nn.Linear(8 + 1, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        
        self.head = nn.Sequential(
            nn.Linear(self.img_dim + 64, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )
        self.loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

    def forward(self, img, target_name, height):
        img_feats = self.base_model(img)
        t_feat = self.target_emb(target_name.long())
        tab_in = torch.cat([t_feat, height.unsqueeze(1)], dim=1)
        tab_feats = self.tabular_net(tab_in)
        return self.head(torch.cat([img_feats, tab_feats], dim=1))

    def training_step(self, batch, batch_idx):
        img, tab, y = batch
        # tab order in Dataset: [target_name, height]
        logits = self(img, tab[:, 0], tab[:, 1])
        loss = self.loss_fn(logits, y)
        
        acc = (logits.argmax(dim=1) == y).float().mean()
        self.log("train_loss", loss, prog_bar=True)
        self.log("train_acc", acc, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        img, tab, y = batch
        logits = self(img, tab[:, 0], tab[:, 1])
        loss = self.loss_fn(logits, y)
        acc = (logits.argmax(dim=1) == y).float().mean()
        self.log("val_loss", loss, prog_bar=True)
        self.log("val_acc", acc, prog_bar=True)

    def configure_optimizers(self):
        # AdamW is better for regularization than standard Adam
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.hparams.lr, weight_decay=0.05)
        # Cosine Annealing with Warm Restarts helps escape local minima
        scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
            optimizer, T_0=5, T_mult=1, eta_min=1e-6
        )
        return [optimizer], [scheduler]

# --- Training Section ---
image_root_dir = "/kaggle/input/csiro-biomass"
train_df, valid_df = train_test_split(train_pd, test_size=0.2, random_state=42, stratify=train_pd["Species"])

datamodule = SpeciesDataModule(train_df=train_df, valid_df=valid_df, root_dir=image_root_dir)
datamodule.setup()

# FIX: Calculate dimensions for the constructor
num_species = len(SPECIES_LE.classes_)
target_count = len(TARGET_LE.classes_)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# FIX: Pass both num_classes AND target_dim
species_model = EfficientNetSpeciesClassifier(
    num_classes=num_species, 
    target_dim=target_count
).to(device)

# 1. Define the safety net
early_stop_callback = EarlyStopping(
    monitor="val_loss",   # or "val_rmse" / "val_mse"
    patience=5,           # If no improvement for 5 epochs, stop.
    mode="min"
)

checkpoint_callback = ModelCheckpoint(
    monitor="val_loss",
    save_top_k=1,
    mode="min"
)

# 2. Configure the trainer with SWA and Clipping
trainer = Trainer(
    accelerator="gpu",
    devices=1,
    max_epochs=30,        # Set higher, but early stopping will control it
    precision="16-mixed",
    gradient_clip_val=1.0, 
    callbacks=[early_stop_callback, checkpoint_callback],
    #use_swa=True         
)

trainer.fit(species_model, datamodule)

# Inference
species_model.eval()
species_model.to(device)

inf_tf = v2.Compose([
    v2.Resize(256), v2.CenterCrop(224), v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

species_results = []
with torch.no_grad():
    for _, row in tqdm(test_pd.iterrows(), total=len(test_pd), desc="Predicting Species"):
        try:
            img_path = os.path.join(image_root_dir, row["image_path"])
            img = Image.open(img_path).convert("RGB")
            img_tensor = inf_tf(img).unsqueeze(0).to(device)
            
            t_idx = safe_encode(TARGET_LE, row["target_name"])
            tab_tensor = torch.tensor([[t_idx, float(row["Height_Ave_cm"])]], dtype=torch.float32).to(device)
            
            logits = species_model(img_tensor, tab_tensor)
            class_idx = logits.argmax(dim=1).item()
            actual_name = SPECIES_LE.inverse_transform([class_idx])[0]
            species_results.append(actual_name)
        except:
            species_results.append(SPECIES_LE.classes_[0])

test_pd["Species"] = species_results
print("✅ Species predictions completed")


=== STEP 2: Species Prediction ===
✅ Loaded EfficientNet-B0 weights for Species Prediction


Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
2026-01-16 10:30:46.541755: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768559446.737106      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768559446.789199      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768559447.258226      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768559447.258255      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid lin

┏━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name        ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ base_model  │ EfficientNet     │  4.0 M │ train │     0 │
│ 1 │ target_emb  │ Embedding        │     48 │ train │     0 │
│ 2 │ tabular_net │ Sequential       │    768 │ train │     0 │
│ 3 │ head        │ Sequential       │  348 K │ train │     0 │
│ 4 │ loss_fn     │ CrossEntropyLoss │      0 │ train │     0 │
└───┴─────────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 4.4 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 4.4 M                                                                                                
Total estimated model params size (MB): 17                                                                         
Modules in train mode: 348                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

`Trainer.fit` stopped: `max_epochs=30` reached.


Predicting Species: 100%|██████████| 5/5 [00:00<00:00, 16.97it/s]

✅ Species predictions completed


In [5]:
# ============================================================================
# CELL 4: STATE PREDICTION
# ============================================================================
from torch.utils.data import WeightedRandomSampler
print("\n=== STEP 3: State Prediction ===")

STATE_LE = LabelEncoder()
STATE_LE.fit(train_pd["State"].astype(str).unique())

class StateDataset(Dataset):
    def __init__(self, df, root_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = os.path.join(self.root_dir, row["image_path"])
        
        try:
            image = Image.open(image_path).convert("RGB")
        except:
            image = Image.new('RGB', (224, 224), (0, 0, 0))

        tabular = torch.tensor([
            float(row["target_name"]), 
            float(row["Height_Ave_cm"]), 
            float(row["Species"])
        ], dtype=torch.float32)

        y = torch.tensor(row["State"], dtype=torch.long)
        
        if self.transform:
            image = self.transform(image)
        return image, tabular, y

class StateDataModule(pl.LightningDataModule):
    def __init__(self, train_df, valid_df, root_dir, batch_size=16, num_workers=2):
        super().__init__()
        self.train_df, self.valid_df = train_df.copy(), valid_df.copy()
        self.root_dir, self.batch_size, self.num_workers = root_dir, batch_size, num_workers

    def setup(self, stage=None):
        for df in [self.train_df, self.valid_df]:
            if not np.issubdtype(df["State"].dtype, np.number):
                df["State"] = df["State"].apply(lambda x: safe_encode(STATE_LE, x))
            if not np.issubdtype(df["target_name"].dtype, np.number):
                df["target_name"] = df["target_name"].apply(lambda x: safe_encode(TARGET_LE, x))
            if not np.issubdtype(df["Species"].dtype, np.number):
                df["Species"] = df["Species"].apply(lambda x: safe_encode(SPECIES_LE, x))

        self.train_tf = v2.Compose([
            v2.RandomResizedCrop(224), v2.RandomHorizontalFlip(), v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])
        self.valid_tf = v2.Compose([
            v2.Resize(256), v2.CenterCrop(224), v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])
        self.train_ds = StateDataset(self.train_df, self.root_dir, self.train_tf)
        self.valid_ds = StateDataset(self.valid_df, self.root_dir, self.valid_tf)

    def train_dataloader(self):
        return DataLoader(self.train_ds, batch_size=self.batch_size, shuffle=True, num_workers=self.num_workers)
    
    def val_dataloader(self):
        return DataLoader(self.valid_ds, batch_size=self.batch_size, shuffle=False, num_workers=self.num_workers)

class EfficientNetStateClassifier(pl.LightningModule):
    def __init__(self, num_classes, target_dim, species_dim, lr=1e-4):
        super().__init__()
        self.save_hyperparameters()
        
        # 1. Base Model with Local Weights
        self.base_model = models.efficientnet_b0(weights=None)
        # ... [Your local weight loading logic remains here] ...

        self.img_dim = self.base_model.classifier[1].in_features
        self.base_model.classifier = nn.Identity()

        # 2. Categorical Embeddings (Crucial for State/Species)
        self.target_emb = nn.Embedding(target_dim + 1, 8)
        self.species_emb = nn.Embedding(species_dim + 1, 12)
        
        # 3. Tabular Branch (8 + 12 + 1 height = 21 inputs)
        self.tabular_net = nn.Sequential(
            nn.Linear(21, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        
        # 4. Classification Head
        self.head = nn.Sequential(
            nn.Linear(self.img_dim + 64, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )
        
        # Use Label Smoothing to combat overfitting
        self.loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

    def forward(self, img, target_name, height, species):
        img_feats = self.base_model(img)
        
        # Embeddings
        t_feat = self.target_emb(target_name.long())
        s_feat = self.species_emb(species.long())
        
        # Combine tabular
        tab_in = torch.cat([t_feat, height.unsqueeze(1), s_feat], dim=1)
        tab_feats = self.tabular_net(tab_in)
        
        combined = torch.cat([img_feats, tab_feats], dim=1)
        return self.head(combined)

    def training_step(self, batch, batch_idx):
        img, tab, y = batch
        # tab order: [target_name, height, species]
        logits = self(img, tab[:, 0], tab[:, 1], tab[:, 2])
        loss = self.loss_fn(logits, y)
        
        # Log Accuracy as well (Better for classification)
        acc = (logits.argmax(dim=1) == y).float().mean()
        self.log("train_loss", loss, prog_bar=True)
        self.log("train_acc", acc, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        img, tab, y = batch
        logits = self(img, tab[:, 0], tab[:, 1], tab[:, 2])
        loss = self.loss_fn(logits, y)
        acc = (logits.argmax(dim=1) == y).float().mean()
        self.log("val_loss", loss, prog_bar=True)
        self.log("val_acc", acc, prog_bar=True)

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.hparams.lr, weight_decay=0.05)
        # Cosine Annealing helps find a more generalized global minimum
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)
        return [optimizer], [scheduler]

# Training
train_df, valid_df = train_test_split(train_pd, test_size=0.2, random_state=42, stratify=train_pd["State"])
datamodule = StateDataModule(train_df=train_df, valid_df=valid_df, root_dir=image_root_dir)
datamodule.setup()

# 1. Calculate dimensions for the embeddings
num_states = len(STATE_LE.classes_)
target_count = len(TARGET_LE.classes_)
species_count = len(SPECIES_LE.classes_)

# 2. Initialize the model with all required arguments
state_model = EfficientNetStateClassifier(
    num_classes=num_states, 
    target_dim=target_count, 
    species_dim=species_count
).to(device)

# 1. Define the safety net
early_stop_callback = EarlyStopping(
    monitor="val_loss",   # or "val_rmse" / "val_mse"
    patience=5,           # If no improvement for 5 epochs, stop.
    mode="min"
)

checkpoint_callback = ModelCheckpoint(
    monitor="val_loss",
    save_top_k=1,
    mode="min"
)

# 2. Configure the trainer with SWA and Clipping
trainer = Trainer(
    accelerator="gpu",
    devices=1,
    max_epochs=30,        # Set higher, but early stopping will control it
    precision="16-mixed",
    gradient_clip_val=1.0, 
    callbacks=[early_stop_callback, checkpoint_callback],
    #use_swa=True         
)
trainer.fit(state_model, datamodule)

# Inference
state_model.eval()
state_model.to(device)

state_results = []
with torch.no_grad():
    for _, row in tqdm(test_pd.iterrows(), total=len(test_pd), desc="Predicting State"):
        try:
            img_path = os.path.join(image_root_dir, row["image_path"])
            img = Image.open(img_path).convert("RGB")
            img_tensor = inf_tf(img).unsqueeze(0).to(device)
            
            t_idx = safe_encode(TARGET_LE, row["target_name"])
            s_idx = safe_encode(SPECIES_LE, row["Species"])
            h_val = float(row["Height_Ave_cm"])
            
            tab_tensor = torch.tensor([[t_idx, h_val, s_idx]], dtype=torch.float32).to(device)
            
            logits = state_model(img_tensor, tab_tensor)
            class_idx = logits.argmax(dim=1).item()
            actual_state = STATE_LE.inverse_transform([class_idx])[0]
            state_results.append(actual_state)
        except:
            state_results.append(STATE_LE.classes_[0])

test_pd["State"] = state_results
print("✅ State predictions completed")


=== STEP 3: State Prediction ===


Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/model_summary/model_summary.py:242: Precision 16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.


┏━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name        ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ base_model  │ EfficientNet     │  4.0 M │ train │     0 │
│ 1 │ target_emb  │ Embedding        │     48 │ train │     0 │
│ 2 │ species_emb │ Embedding        │    192 │ train │     0 │
│ 3 │ tabular_net │ Sequential       │  1.5 K │ train │     0 │
│ 4 │ head        │ Sequential       │  345 K │ train │     0 │
│ 5 │ loss_fn     │ CrossEntropyLoss │      0 │ train │     0 │
└───┴─────────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 4.4 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 4.4 M                                                                                                
Total estimated model params size (MB): 17                                                                         
Modules in train mode: 349                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

Predicting State: 100%|██████████| 5/5 [00:00<00:00, 16.84it/s]

✅ State predictions completed


In [6]:
# ============================================================================
# CELL 5: NDVI PREDICTION
# ============================================================================
print("\n=== STEP 4: NDVI Prediction ===")

def image_to_mask(image_path):
    """Applies HSV masking to isolate green vegetation."""
    image = cv2.imread(image_path)
    if image is None:
        return np.zeros((224, 224, 3), dtype=np.uint8)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)
    lower_green = np.array([35, 40, 40])
    upper_green = np.array([85, 255, 255])
    mask = cv2.inRange(hsv, lower_green, upper_green)
    return cv2.bitwise_and(image, image, mask=mask)

class PreGSSHDataset(Dataset):
    def __init__(self, df, root_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.root_dir, row["image_path"])
        masked_img = image_to_mask(img_path)
        
        s_idx = safe_encode(SPECIES_LE, row["Species"])
        t_idx = safe_encode(TARGET_LE, row["target_name"])
        ndvi_val = row["Pre_GSHH_NDVI"] if "Pre_GSHH_NDVI" in self.df.columns else 0.0
        
        tabular = torch.tensor([
            float(s_idx),
            float(ndvi_val),
            float(row["Height_Ave_cm"]),
            float(t_idx)
        ], dtype=torch.float32)

        target = torch.tensor([ndvi_val], dtype=torch.float32)

        if self.transform:
            image = self.transform(masked_img)
        else:
            image = torch.tensor(masked_img).permute(2, 0, 1).float()

        return image, tabular, target

class PreGSSHDataModule(pl.LightningDataModule):
    def __init__(self, train_df, valid_df, root_dir, batch=16):
        super().__init__()
        self.train_df, self.valid_df = train_df, valid_df
        self.root_dir, self.batch = root_dir, batch
        self.tfs = v2.Compose([
            v2.ToImage(), v2.Resize((224, 224)),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])

    def setup(self, stage=None):
        self.train_ds = PreGSSHDataset(self.train_df, self.root_dir, self.tfs)
        self.valid_ds = PreGSSHDataset(self.valid_df, self.root_dir, self.tfs)

    def train_dataloader(self):
        return DataLoader(self.train_ds, batch_size=self.batch, shuffle=True, num_workers=2)
    
    def val_dataloader(self):
        return DataLoader(self.valid_ds, batch_size=self.batch, num_workers=2)

import torch.nn as nn
import torch.nn.functional as F

class PreGSSHModel(pl.LightningModule):
    def __init__(self, species_dim, target_dim, lr=1e-4):
        super().__init__()
        self.save_hyperparameters()
        
        # 1. Backbone with Local Weights
        self.resnet = models.resnet50(weights=None)
        local_weights = '/kaggle/input/se_resnet50/pytorch/default/1/se_resnet50-ce0d4300.pth'
        if os.path.exists(local_weights):
            state_dict = torch.load(local_weights, map_location="cpu")
            # Cleaning keys for standard ResNet50 compatibility
            new_state_dict = {k.replace("layer0.", "").replace("last_linear", "fc"): v for k, v in state_dict.items()}
            self.resnet.load_state_dict(new_state_dict, strict=False)
            print("✅ Loaded SE-ResNet50 local weights")
        
        self.img_dim = self.resnet.fc.in_features
        self.resnet.fc = nn.Identity()

        # 2. Categorical Embeddings
        self.species_emb = nn.Embedding(species_dim + 1, 8)
        self.target_emb = nn.Embedding(target_dim + 1, 4)
        
        # 3. Tabular Branch (8 + 4 + 1 height = 13 inputs)
        self.tabular_net = nn.Sequential(
            nn.Linear(13, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        
        # 4. Final Head with Sigmoid
        self.head = nn.Sequential(
            nn.Linear(self.img_dim + 64, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Linear(128, 1),
            nn.Sigmoid() # Forces output to [0, 1] range
        )
        
        self.loss_fn = nn.HuberLoss(delta=0.1) # Smooth L1/Huber is great for 0-1 ranges

    def forward(self, img, species, target_name, height):
        img_feats = self.resnet(img)
        
        # Embeddings
        s_feat = self.species_emb(species.long())
        t_feat = self.target_emb(target_name.long())
        
        # Combine tabular (Species + Target + Height)
        tab_in = torch.cat([s_feat, t_feat, height.unsqueeze(1)], dim=1)
        tab_feats = self.tabular_net(tab_in)
        
        combined = torch.cat([img_feats, tab_feats], dim=1)
        return self.head(combined)

    def training_step(self, batch, batch_idx):
        img, tab, y = batch
        # tab order based on Dataset: [species, ndvi, height, target]
        preds = self(img, tab[:, 0], tab[:, 3], tab[:, 2])
        loss = self.loss_fn(preds, y)
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        img, tab, y = batch
        preds = self(img, tab[:, 0], tab[:, 3], tab[:, 2])
        loss = F.mse_loss(preds, y) # Track MSE for validation
        self.log("val_mse", loss, prog_bar=True)

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.hparams.lr, weight_decay=0.01)
        scheduler = torch.optim.lr_scheduler.OneCycleLR(
            optimizer, max_lr=self.hparams.lr, 
            total_steps=self.trainer.estimated_stepping_batches
        )
        return [optimizer], [{"scheduler": scheduler, "interval": "step"}]

# Training
train_df, valid_df = train_test_split(train_pd, test_size=0.2, random_state=42)
datamodule = PreGSSHDataModule(train_df, valid_df, image_root_dir)

inf_tfs = v2.Compose([
    v2.ToImage(), v2.Resize((224, 224)),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# --- Training Section ---
species_count = len(SPECIES_LE.classes_)
target_count = len(TARGET_LE.classes_)

# Initialize with required dims
ndvi_model = PreGSSHModel(species_dim=species_count, target_dim=target_count)

# 1. Define the safety net
early_stop_callback = EarlyStopping(
    monitor="val_mse",   # or "val_rmse" / "val_mse"
    patience=5,           # If no improvement for 5 epochs, stop.
    mode="min"
)

checkpoint_callback = ModelCheckpoint(
    monitor="val_mse",
    save_top_k=1,
    mode="min"
)

# 2. Configure the trainer with SWA and Clipping
trainer = Trainer(
    accelerator="gpu",
    devices=1,
    max_epochs=30,        # Set higher, but early stopping will control it
    precision="16-mixed",
    gradient_clip_val=1.0, 
    callbacks=[early_stop_callback, checkpoint_callback],
    #use_swa=True         
)
trainer.fit(ndvi_model, datamodule)

# --- Final Inference Section ---
ndvi_model.eval()
ndvi_model.to(device)

ndvi_results = []
with torch.no_grad():
    for _, row in tqdm(test_pd.iterrows(), total=len(test_pd), desc="Predicting NDVI"):
        try:
            img_path = os.path.join(image_root_dir, row["image_path"])
            masked = image_to_mask(img_path)
            img_tensor = inf_tfs(masked).unsqueeze(0).to(device)
            
            # Prepare separate inputs for the forward method
            s_idx = torch.tensor([safe_encode(SPECIES_LE, row["Species"])]).to(device)
            t_idx = torch.tensor([safe_encode(TARGET_LE, row["target_name"])]).to(device)
            h_val = torch.tensor([float(row["Height_Ave_cm"])]).to(device)
            
            # FIX: Call forward with separate arguments as defined in your class
            pred_ndvi = ndvi_model(img_tensor, s_idx, t_idx, h_val).item()
            
            # Ensure the output is within the Sigmoid range (though Sigmoid handles this)
            ndvi_results.append(float(pred_ndvi))
        except Exception as e:
            ndvi_results.append(0.0)

test_pd["Pre_GSHH_NDVI"] = ndvi_results

print("✅ NDVI predictions completed")


=== STEP 4: NDVI Prediction ===


Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loading `train_dataloader` to estimate number of stepping batches.


✅ Loaded SE-ResNet50 local weights


/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/model_summary/model_summary.py:242: Precision 16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.


┏━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name        ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ resnet      │ ResNet     │ 23.5 M │ train │     0 │
│ 1 │ species_emb │ Embedding  │    128 │ train │     0 │
│ 2 │ target_emb  │ Embedding  │     24 │ train │     0 │
│ 3 │ tabular_net │ Sequential │  1.0 K │ train │     0 │
│ 4 │ head        │ Sequential │  270 K │ train │     0 │
│ 5 │ loss_fn     │ HuberLoss  │      0 │ train │     0 │
└───┴─────────────┴────────────┴────────┴───────┴───────┘

Trainable params: 23.8 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 23.8 M                                                                                               
Total estimated model params size (MB): 95                                                                         
Modules in train mode: 165                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

Predicting NDVI: 100%|██████████| 5/5 [00:00<00:00,  8.90it/s]

✅ NDVI predictions completed


In [7]:
# ============================================================================
# CELL 6: FINAL BIOMASS PREDICTION
# ============================================================================
import torch.nn as nn
import torch.nn.functional as F

print("\n=== STEP 5: Final Biomass Prediction ===")

# Fit the final encoder for target_name if not already done
TARGET_NAME_LE = LabelEncoder()
TARGET_NAME_LE.fit(train_pd["target_name"].astype(str).unique())

class BiomassDataset(Dataset):
    def __init__(self, df, root_dir, transform=None, is_train=True):
        self.df = df.reset_index(drop=True)
        self.root_dir = root_dir
        self.transform = transform
        self.is_train = is_train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = os.path.join(self.root_dir, row["image_path"])
        
        try:
            img = Image.open(image_path).convert("RGB")
        except:
            img = Image.new('RGB', (224, 224), (0, 0, 0))
            
        if self.transform:
            img = self.transform(img)

        # Build tabular feature vector from predicted and metadata columns
        tab = torch.tensor([
            float(safe_encode(STATE_LE, row["State"])),
            float(safe_encode(SPECIES_LE, row["Species"])),
            float(row["Pre_GSHH_NDVI"]),
            float(row["Height_Ave_cm"]),
            float(safe_encode(TARGET_NAME_LE, row["target_name"]))
        ], dtype=torch.float32)

        if self.is_train:
            # The 'target' column is the actual biomass in the training set
            target = torch.tensor(row["target"], dtype=torch.float32)
            return img, tab, target
        else:
            return img, tab


class BiomassLightningModel(pl.LightningModule):
    def __init__(self, state_dim, species_dim, target_name_dim, lr=1e-3):
        super().__init__()
        self.save_hyperparameters()

        # 1. Load Backbone Weights
        self.resnet = models.resnet50(weights=None)
        local_weights = '/kaggle/input/se_resnet50/pytorch/default/1/se_resnet50-ce0d4300.pth'
        
        if os.path.exists(local_weights):
            state_dict = torch.load(local_weights, map_location="cpu")
            # Cleaning keys for standard ResNet50
            new_state_dict = {k.replace("layer0.", "").replace("last_linear", "fc"): v for k, v in state_dict.items()}
            self.resnet.load_state_dict(new_state_dict, strict=False)
            print("✅ SE-ResNet50 Weights Loaded Successfully")

        self.img_dim = self.resnet.fc.in_features
        self.resnet.fc = nn.Identity()

        # 2. Define New Layers
        self.state_emb = nn.Embedding(state_dim + 1, 8)
        self.species_emb = nn.Embedding(species_dim + 1, 16)
        self.target_name_emb = nn.Embedding(target_name_dim + 1, 8)
        
        self.tab_net = nn.Sequential(
            nn.Linear(34, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64)
        )
        
        self.head = nn.Sequential(
            nn.Linear(self.img_dim + 64, 512),
            nn.BatchNorm1d(512), # Added for stability
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 1)
        )

        # 3. SMART Initialization (Fixes the bug)
        # We only initialize our custom layers, NOT the resnet we just loaded
        for m in [self.state_emb, self.species_emb, self.target_name_emb, self.tab_net, self.head]:
            for layer in m.modules():
                if isinstance(layer, (nn.Linear, nn.Conv2d)):
                    nn.init.kaiming_normal_(layer.weight, mode='fan_out', nonlinearity='relu')
                elif isinstance(layer, nn.BatchNorm1d):
                    nn.init.constant_(layer.weight, 1)
                    nn.init.constant_(layer.bias, 0)

    def forward(self, img, state, species, target_name, ndvi_height):
        img_feats = self.resnet(img)
        
        s_emb = self.state_emb(state.long())
        sp_emb = self.species_emb(species.long())
        t_emb = self.target_name_emb(target_name.long())
        
        tab_combined = torch.cat([s_emb, sp_emb, t_emb, ndvi_height], dim=1)
        tab_feats = self.tab_net(tab_combined)
        
        combined = torch.cat([img_feats, tab_feats], dim=1)
        return self.head(combined).squeeze(1)

    def training_step(self, batch, batch_idx):
        img, tab, y = batch
        
        # Split tab back into components for the forward pass
        # (Assuming you update the Dataset to return these separately)
        state, species, ndvi, height, t_name = tab[:,0], tab[:,1], tab[:,2], tab[:,3], tab[:,4]
        ndvi_height = tab[:, 2:4]
        
        preds = self(img, state, species, t_name, ndvi_height)
        loss = F.huber_loss(preds, y) # Huber is more stable than MSE for scratch training
        self.log("train_loss", loss)
        return loss

    def validation_step(self, batch, batch_idx):
        img, tab, y = batch
        # Same unpacking as training_step
        state, species, ndvi, height, t_name = tab[:,0], tab[:,1], tab[:,2], tab[:,3], tab[:,4]
        ndvi_height = tab[:, 2:4]
        
        preds = self(img, state, species, t_name, ndvi_height)
        
        # We use MSE for the monitor, but you can also log RMSE
        loss = F.mse_loss(preds, y)
        self.log("val_mse", loss, prog_bar=True)
        return loss

    def configure_optimizers(self):
        # Use OneCycleLR: It’s the gold standard for training from scratch fast
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.hparams.lr, weight_decay=0.05)
        scheduler = torch.optim.lr_scheduler.OneCycleLR(
            optimizer, max_lr=self.hparams.lr, 
            total_steps=self.trainer.estimated_stepping_batches
        )
        return [optimizer], [{"scheduler": scheduler, "interval": "step"}]

# --- Prepare Data ---
train_df, valid_df = train_test_split(train_pd, test_size=0.15, random_state=42)

# Using the same transforms as previous steps
biomass_tfs = v2.Compose([
    v2.Resize((224, 224)),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_ds = BiomassDataset(train_df, image_root_dir, biomass_tfs, is_train=True)
valid_ds = BiomassDataset(valid_df, image_root_dir, biomass_tfs, is_train=True)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=2)
valid_loader = DataLoader(valid_ds, batch_size=16, num_workers=2)


# --- Calculate Dimensions ---
num_states = len(STATE_LE.classes_)
num_species = len(SPECIES_LE.classes_)
num_targets = len(TARGET_NAME_LE.classes_)

# --- Initialize Model ---
biomass_model = BiomassLightningModel(
    state_dim=num_states, 
    species_dim=num_species, 
    target_name_dim=num_targets
)

# --- Train Model ---
# Added ModelCheckpoint to save the best version
from pytorch_lightning.callbacks import ModelCheckpoint
checkpoint_cb = ModelCheckpoint(monitor="train_loss", mode="min", save_top_k=1)

# 1. Early Stopping: Prevents overfitting by stopping when validation stops improving
early_stop_callback = EarlyStopping(
    monitor="val_mse",  
    patience=5,           
    mode="min",
    verbose=True
)

# 2. Checkpoint: Saves the version of the model that had the BEST validation score
checkpoint_callback = ModelCheckpoint(
    dirpath="/kaggle/working/checkpoints",
    filename="best-biomass-model",
    monitor="val_mse",
    save_top_k=1,
    mode="min"
)

# --- Configure the Trainer ---
trainer = Trainer(
    accelerator="gpu",
    devices=1,
    max_epochs=40, # Higher max, but EarlyStopping will stop it sooner
    precision="16-mixed",
    gradient_clip_val=1.0, 
    callbacks=[early_stop_callback, checkpoint_callback],
    #use_swa=True         
)

trainer.fit(biomass_model, train_loader, valid_loader)

# --- Final Inference ---
biomass_model.eval()
biomass_model.to(device)
final_results = []

with torch.no_grad():
    for _, row in tqdm(test_pd.iterrows(), total=len(test_pd), desc="Calculating Biomass"):
        try:
            img_path = os.path.join(image_root_dir, row["image_path"])
            img = Image.open(img_path).convert("RGB")
            img_tensor = biomass_tfs(img).unsqueeze(0).to(device)
            
            # Extract values for separate arguments
            s_val = torch.tensor([safe_encode(STATE_LE, row["State"])]).to(device)
            sp_val = torch.tensor([safe_encode(SPECIES_LE, row["Species"])]).to(device)
            t_val = torch.tensor([safe_encode(TARGET_NAME_LE, row["target_name"])]).to(device)
            
            # NDVI and Height combined into one tensor (shape: 1, 2)
            nh_val = torch.tensor([[float(row["Pre_GSHH_NDVI"]), float(row["Height_Ave_cm"])]], dtype=torch.float32).to(device)
            
            # FIX: Call forward with individual arguments to match definition
            pred_biomass = biomass_model(img_tensor, s_val, sp_val, t_val, nh_val).item()
            
            final_results.append(max(0.0, pred_biomass))
        except Exception as e:
            final_results.append(0.0)

# ============= CREATE SUBMISSION =============
submission_df = pd.DataFrame({
    "sample_id": test_pd_original["sample_id"],
    "target": final_results
})

submission_df.to_csv('/kaggle/working/submission.csv', index=False)

print(f"\n✅ Pipeline Complete! Submission saved to submission.csv")
print(submission_df.head(10))


=== STEP 5: Final Biomass Prediction ===


Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loading `train_dataloader` to estimate number of stepping batches.


✅ SE-ResNet50 Weights Loaded Successfully


/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/model_summary/model_summary.py:242: Precision 16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.


┏━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name            ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ resnet          │ ResNet     │ 23.5 M │ train │     0 │
│ 1 │ state_emb       │ Embedding  │     40 │ train │     0 │
│ 2 │ species_emb     │ Embedding  │    256 │ train │     0 │
│ 3 │ target_name_emb │ Embedding  │     48 │ train │     0 │
│ 4 │ tab_net         │ Sequential │ 13.0 K │ train │     0 │
│ 5 │ head            │ Sequential │  1.1 M │ train │     0 │
└───┴─────────────────┴────────────┴────────┴───────┴───────┘

Trainable params: 24.6 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 24.6 M                                                                                               
Total estimated model params size (MB): 98                                                                         
Modules in train mode: 166                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

Metric val_mse improved. New best score: 5345.257
Metric val_mse improved by 3779.488 >= min_delta = 0.0. New best score: 1565.768
Metric val_mse improved by 931.382 >= min_delta = 0.0. New best score: 634.386
Metric val_mse improved by 396.128 >= min_delta = 0.0. New best score: 238.258
Metric val_mse improved by 94.655 >= min_delta = 0.0. New best score: 143.603
Monitored metric val_mse did not improve in the last 5 records. Best score: 143.603. Signaling Trainer to stop.


Calculating Biomass: 100%|██████████| 5/5 [00:00<00:00, 15.66it/s]


✅ Pipeline Complete! Submission saved to submission.csv
                    sample_id      target
0  ID1001187975__Dry_Clover_g   94.774918
1    ID1001187975__Dry_Dead_g  102.623192
2   ID1001187975__Dry_Green_g  178.068741
3   ID1001187975__Dry_Total_g  186.722763
4         ID1001187975__GDM_g  179.231155
